# Semantic Search

**Semantic search** finds documents that are *meaningfully similar* to a query, not just those that share exact keywords. It works by embedding both the query and the documents into the same vector space, then finding nearest neighbours.

In this notebook we:
1. Build a document corpus
2. Embed it with `sentence-transformers`
3. Index it with **FAISS** (via the reusable `src/VectorSearch` class)
4. Run queries and rank results
5. Compare semantic search vs exact keyword matching

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers faiss-cpu pillow -q

## Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))  # make src/ importable

from sentence_transformers import SentenceTransformer
import numpy as np
import torch

print("Libraries loaded.")

## 1. Build the Document Corpus

In [ ]:
# Load from datasets/example_documents.txt (project corpus)
corpus_path = os.path.join(os.getcwd(), "..", "datasets", "example_documents.txt")
with open(corpus_path) as f:
    file_docs = [line.strip() for line in f if line.strip()]

# Extend with additional documents for a richer demo
extra_docs = [
    "Natural language processing enables computers to understand human language.",
    "Convolutional neural networks excel at image recognition tasks.",
    "Transformers use self-attention to process sequences in parallel.",
    "BERT is a bidirectional transformer pre-trained on large text corpora.",
    "Reinforcement learning trains agents through reward and penalty signals.",
    "Data preprocessing is a critical step before training any model.",
    "GPT models are autoregressive and generate text token by token.",
    "Embeddings map words and sentences to dense numerical vectors.",
]

documents = file_docs + extra_docs
print(f"Corpus size: {len(documents)} documents\n")
for i, doc in enumerate(documents):
    print(f"  [{i:2d}] {doc}")

## 2. Embed the Corpus

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = model.encode(documents, show_progress_bar=True)

print(f"\nEmbedding matrix shape: {doc_embeddings.shape}")
print(f"Each document → vector of {doc_embeddings.shape[1]} dimensions")

## 3. Build the FAISS Index with `src/VectorSearch`

`VectorSearch` (in `src/vector_search.py`) wraps a FAISS `IndexFlatL2` index. We build the index once, then query it in O(n) time — or much faster with approximate indices for large corpora.

In [ ]:
from vector_search import VectorSearch

dimension = doc_embeddings.shape[1]
vs = VectorSearch(dimension)
vs.add(doc_embeddings.astype("float32"))

print(f"FAISS index built with {vs.index.ntotal} vectors (dimension={dimension})")

## 4. Run Semantic Search Queries

In [ ]:
def semantic_search(query, top_k=3):
    q_emb = model.encode([query]).astype("float32")
    distances, indices = vs.search(q_emb[0], k=top_k)
    print(f"Query: '{query}'\n")
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
        print(f"  {rank}. [L2 dist={dist:.3f}] {documents[idx]}")
    print()

queries = [
    "How do computers learn from examples?",
    "What technology is used for image recognition?",
    "How does GPT generate text?",
    "What is the role of attention in transformers?",
]

for q in queries:
    semantic_search(q)

## 5. Semantic Search vs Keyword Search

Keyword search matches exact words. Semantic search matches *meaning*. Notice that semantic search finds relevant results even when no keywords overlap.

In [ ]:
def keyword_search(query, corpus, top_k=3):
    query_words = set(query.lower().split())
    scored = []
    for doc in corpus:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)
        scored.append((overlap, doc))
    scored.sort(reverse=True)
    return scored[:top_k]

# This query has NO exact keyword overlap with the corpus
query = "How do machines acquire knowledge from data?"

print("=== KEYWORD SEARCH ===")
kw_results = keyword_search(query, documents)
for score, doc in kw_results:
    print(f"  [overlap={score}] {doc}")

print("\n=== SEMANTIC SEARCH ===")
semantic_search(query)

print("Keyword search struggles with paraphrasing; semantic search understands intent.")